In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
OUT_DIR     = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use")
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_ROUNDS = 40
MIN_PRIOR_ROUNDS = 20
MIN_EVENT_ROUNDS = 2
MIN_COURSE_PLAYER_EVENTS = 200
SHRINK_K = 800

ABS_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026.parquet"
ABS_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026.csv"
REL_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.parquet"
REL_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.csv"

needed = {
    "tour","year","event_id","event_name","course_num","course_name",
    "dg_id","player_name","round_num","round_date",
    "sg_total","sg_ott","sg_app","sg_arg","sg_putt",
}
df = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in needed, low_memory=False)

num_cols = ["year","event_id","course_num","dg_id","round_num","sg_total","sg_ott","sg_app","sg_arg","sg_putt"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["tour","year","event_id","course_num","dg_id","round_num","sg_total"]).copy()
df["tour"] = df["tour"].astype(str)
df["year"] = df["year"].astype(int)
df["event_id"] = df["event_id"].astype(int)
df["course_num"] = df["course_num"].astype(int)
df["dg_id"] = df["dg_id"].astype(int)
df["round_num"] = df["round_num"].astype(int)

skill_cols = ["sg_ott","sg_app","sg_arg","sg_putt"]
for c in skill_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["tour","dg_id","year","event_id","round_num"]).copy()
df["pt_round_idx"] = df.groupby(["tour","dg_id"]).cumcount()

event_first_idx = (
    df.groupby(["tour","year","event_id","dg_id"], as_index=False)["pt_round_idx"]
      .min()
      .rename(columns={"pt_round_idx":"event_first_pt_idx"})
)

event_outcome = (
    df.groupby(["tour","year","event_id","course_num","dg_id"], as_index=False)
      .agg(event_sg_total=("sg_total","sum"),
           event_rounds=("sg_total","size"))
      .merge(event_first_idx, on=["tour","year","event_id","dg_id"], how="left")
)

names = df[["tour","dg_id","player_name"]].drop_duplicates()
course_name_map = (
    df.dropna(subset=["course_name"])
      .groupby("course_num")["course_name"]
      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
      .to_dict()
)

def fit_one_course(course_num: int) -> dict:
    eo = event_outcome[event_outcome["course_num"] == int(course_num)].copy()
    if eo.empty:
        return {"course_num": int(course_num), "status": "no rows"}

    rows = []
    for r in eo.itertuples(index=False):
        tour = r.tour
        year = int(r.year)
        event_id = int(r.event_id)
        dg_id = int(r.dg_id)
        y = float(r.event_sg_total)
        erounds = int(r.event_rounds)
        first_idx = r.event_first_pt_idx

        if pd.isna(first_idx):
            continue
        if erounds < MIN_EVENT_ROUNDS:
            continue

        hist = df[
            (df["tour"] == tour) &
            (df["dg_id"] == dg_id) &
            (df["pt_round_idx"] < first_idx)
        ].tail(WINDOW_ROUNDS)

        if hist["sg_total"].notna().sum() < MIN_PRIOR_ROUNDS:
            continue

        feats = {}
        ok = True
        for c in skill_cols:
            v = hist[c].mean()
            if pd.isna(v):
                ok = False
                break
            feats[f"{c}_pre"] = float(v)
        if not ok:
            continue

        feats.update({
            "tour": tour,
            "year": year,
            "event_id": event_id,
            "course_num": int(course_num),
            "dg_id": dg_id,
            "event_sg_total": y,
            "event_rounds": erounds,
            "prior_rounds_used": int(hist.shape[0]),
        })
        rows.append(feats)

    pe = pd.DataFrame(rows)
    n = int(len(pe))
    if n < MIN_COURSE_PLAYER_EVENTS:
        return {
            "course_num": int(course_num),
            "course_name": course_name_map.get(int(course_num)),
            "status": f"insufficient ({n})",
            "n_player_events": n,
        }

    pe = pe.merge(names, on=["tour","dg_id"], how="left")

    need_pre = [f"{c}_pre" for c in skill_cols]
    pe = pe.dropna(subset=need_pre + ["event_sg_total"]).copy()

    mu = pe[need_pre].mean()
    sd = pe[need_pre].std(ddof=0).replace(0, np.nan)

    for c in need_pre:
        pe[f"z_{c}"] = (pe[c] - mu[c]) / sd[c]

    zcols = [f"z_{c}" for c in need_pre]
    pe = pe.dropna(subset=zcols).copy()

    X = sm.add_constant(pe[zcols])
    y = pe["event_sg_total"]
    m = sm.OLS(y, X).fit()

    beta_ott  = float(m.params.get("z_sg_ott_pre", np.nan))
    beta_app  = float(m.params.get("z_sg_app_pre", np.nan))
    beta_arg  = float(m.params.get("z_sg_arg_pre", np.nan))
    beta_putt = float(m.params.get("z_sg_putt_pre", np.nan))

    w = n / (n + SHRINK_K)
    beta_ott_s  = w * beta_ott
    beta_app_s  = w * beta_app
    beta_arg_s  = w * beta_arg
    beta_putt_s = w * beta_putt

    pred = float(np.sqrt(beta_ott_s**2 + beta_app_s**2 + beta_arg_s**2 + beta_putt_s**2))
    abs_sum = abs(beta_ott_s) + abs(beta_app_s) + abs(beta_arg_s) + abs(beta_putt_s)

    if abs_sum == 0:
        imp_ott = imp_app = imp_arg = imp_putt = np.nan
    else:
        imp_ott  = abs(beta_ott_s)  / abs_sum
        imp_app  = abs(beta_app_s)  / abs_sum
        imp_arg  = abs(beta_arg_s)  / abs_sum
        imp_putt = abs(beta_putt_s) / abs_sum

    return {
        "course_num": int(course_num),
        "course_name": course_name_map.get(int(course_num)),
        "status": "ok",
        "n_player_events": n,
        "r2": float(m.rsquared),
        "randomness": float(1.0 - m.rsquared),  # higher = more random (your requested identifier)
        "beta_ott": beta_ott,
        "beta_app": beta_app,
        "beta_arg": beta_arg,
        "beta_putt": beta_putt,
        "beta_ott_shrunk": beta_ott_s,
        "beta_app_shrunk": beta_app_s,
        "beta_arg_shrunk": beta_arg_s,
        "beta_putt_shrunk": beta_putt_s,
        "predictability": pred,
        "imp_ott": imp_ott,
        "imp_app": imp_app,
        "imp_arg": imp_arg,
        "imp_putt": imp_putt,
    }

all_courses = sorted(event_outcome["course_num"].dropna().unique().tolist())
print(f"Courses to evaluate: {len(all_courses)} (will keep those with >= {MIN_COURSE_PLAYER_EVENTS} player-events)")

results = []
for c in all_courses:
    results.append(fit_one_course(int(c)))

abs_df = pd.DataFrame(results)

abs_ok = abs_df[abs_df["status"] == "ok"].copy().reset_index(drop=True)

abs_ok.to_parquet(ABS_PARQUET, index=False)
abs_ok.to_csv(ABS_CSV, index=False)

print(f"\nOVERWROTE absolute course weights:\n- {ABS_PARQUET}\n- {ABS_CSV}")
print(f"Kept {len(abs_ok)} courses with status=ok.\n")
display(abs_ok.sort_values("predictability", ascending=False).head(20))

rel = abs_ok.copy()

rel_cols = ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk","predictability","randomness","r2"]
for col in rel_cols:
    mu = rel[col].mean()
    sd = rel[col].std(ddof=0)
    rel[f"{col}_z"] = (rel[col] - mu) / sd if sd and sd > 0 else np.nan
    rel[f"{col}_pct"] = rel[col].rank(pct=True)

for col in ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk"]:
    cmin = rel[col].min()
    cmax = rel[col].max()
    rel[f"{col}_minmax01"] = (rel[col] - cmin) / (cmax - cmin) if cmax > cmin else np.nan

rel.to_parquet(REL_PARQUET, index=False)
rel.to_csv(REL_CSV, index=False)

print(f"\nOVERWROTE relative course weights:\n- {REL_PARQUET}\n- {REL_CSV}")
display(rel.sort_values("randomness", ascending=False).head(20))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
OUT_DIR     = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use")

WINDOW_ROUNDS = 40
MIN_PRIOR_ROUNDS = 20
MIN_EVENT_ROUNDS = 2
MIN_COURSE_PLAYER_EVENTS = 200
SHRINK_K = 800  # shrink betas toward global mean (computed across courses after fit)

ABS_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026.parquet"
ABS_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026.csv"
REL_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.parquet"
REL_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.csv"

needed = {
    "tour","year","event_id","event_name","course_num","course_name",
    "dg_id","player_name","round_num","round_date",
    "sg_total","sg_ott","sg_app","sg_arg","sg_putt",
}
df = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in needed, low_memory=False)

num_cols = ["year","event_id","course_num","dg_id","round_num","sg_total","sg_ott","sg_app","sg_arg","sg_putt"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["tour","year","event_id","course_num","dg_id","round_num","sg_total"]).copy()
df["tour"] = df["tour"].astype(str)
df["year"] = df["year"].astype(int)
df["event_id"] = df["event_id"].astype(int)
df["course_num"] = df["course_num"].astype(int)
df["dg_id"] = df["dg_id"].astype(int)
df["round_num"] = df["round_num"].astype(int)

skill_cols = ["sg_ott","sg_app","sg_arg","sg_putt"]
for c in skill_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["tour","dg_id","year","event_id","round_num"]).copy()
df["pt_round_idx"] = df.groupby(["tour","dg_id"]).cumcount()

event_first_idx = (
    df.groupby(["tour","year","event_id","dg_id"], as_index=False)["pt_round_idx"]
      .min()
      .rename(columns={"pt_round_idx":"event_first_pt_idx"})
)

event_outcome = (
    df.groupby(["tour","year","event_id","course_num","dg_id"], as_index=False)
      .agg(event_sg_total=("sg_total","sum"),
           event_rounds=("sg_total","size"))
      .merge(event_first_idx, on=["tour","year","event_id","dg_id"], how="left")
)

names = df[["tour","dg_id","player_name"]].drop_duplicates()
course_name_map = (
    df.dropna(subset=["course_name"])
      .groupby("course_num")["course_name"]
      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
      .to_dict()
)

rows = []
eo = event_outcome.copy()

for r in eo.itertuples(index=False):
    tour = r.tour
    year = int(r.year)
    event_id = int(r.event_id)
    course_num = int(r.course_num)
    dg_id = int(r.dg_id)
    y = float(r.event_sg_total)
    erounds = int(r.event_rounds)
    first_idx = r.event_first_pt_idx

    if pd.isna(first_idx):
        continue
    if erounds < MIN_EVENT_ROUNDS:
        continue

    hist = df[
        (df["tour"] == tour) &
        (df["dg_id"] == dg_id) &
        (df["pt_round_idx"] < first_idx)
    ].tail(WINDOW_ROUNDS)

    if hist["sg_total"].notna().sum() < MIN_PRIOR_ROUNDS:
        continue

    feats = {}
    ok = True
    for c in skill_cols:
        v = hist[c].mean()
        if pd.isna(v):
            ok = False
            break
        feats[f"{c}_pre"] = float(v)
    if not ok:
        continue

    feats.update({
        "tour": tour,
        "year": year,
        "event_id": event_id,
        "course_num": course_num,
        "dg_id": dg_id,
        "event_sg_total": y,
        "event_rounds": erounds,
        "prior_rounds_used": int(hist.shape[0]),
    })
    rows.append(feats)

pe = pd.DataFrame(rows).merge(names, on=["tour","dg_id"], how="left")
print(f"Global player-event rows built: {len(pe):,}")

pre_cols = [f"{c}_pre" for c in skill_cols]
mu = pe[pre_cols].mean()
sd = pe[pre_cols].std(ddof=0).replace(0, np.nan)

for c in pre_cols:
    pe[f"z_{c}"] = (pe[c] - mu[c]) / sd[c]

zcols = [f"z_{c}" for c in pre_cols]
pe = pe.dropna(subset=zcols + ["event_sg_total"]).copy()

def fit_course(course_num: int, g: pd.DataFrame) -> dict:
    n = int(len(g))
    if n < MIN_COURSE_PLAYER_EVENTS:
        return {
            "course_num": int(course_num),
            "course_name": course_name_map.get(int(course_num)),
            "status": f"insufficient ({n})",
            "n_player_events": n,
        }

    X = sm.add_constant(g[zcols])
    y = g["event_sg_total"]
    m = sm.OLS(y, X).fit()

    beta_ott  = float(m.params.get("z_sg_ott_pre", np.nan))
    beta_app  = float(m.params.get("z_sg_app_pre", np.nan))
    beta_arg  = float(m.params.get("z_sg_arg_pre", np.nan))
    beta_putt = float(m.params.get("z_sg_putt_pre", np.nan))

    return {
        "course_num": int(course_num),
        "course_name": course_name_map.get(int(course_num)),
        "status": "ok",
        "n_player_events": n,
        "r2": float(m.rsquared),
        "randomness": float(1.0 - m.rsquared),
        "beta_ott": beta_ott,
        "beta_app": beta_app,
        "beta_arg": beta_arg,
        "beta_putt": beta_putt,
    }

course_rows = []
for cnum, g in pe.groupby("course_num", sort=False):
    course_rows.append(fit_course(int(cnum), g))

abs_df = pd.DataFrame(course_rows)
abs_ok = abs_df[abs_df["status"] == "ok"].copy().reset_index(drop=True)
print(f"Courses kept (status=ok): {len(abs_ok)}")

beta_cols = ["beta_ott","beta_app","beta_arg","beta_putt"]
global_beta_means = abs_ok[beta_cols].mean()

w = abs_ok["n_player_events"] / (abs_ok["n_player_events"] + SHRINK_K)
for b in beta_cols:
    abs_ok[f"{b}_shrunk"] = w * abs_ok[b] + (1 - w) * float(global_beta_means[b])

abs_ok["predictability"] = np.sqrt(
    abs_ok["beta_ott_shrunk"]**2
    + abs_ok["beta_app_shrunk"]**2
    + abs_ok["beta_arg_shrunk"]**2
    + abs_ok["beta_putt_shrunk"]**2
)

abs_sum = (
    abs_ok[["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk"]]
    .abs()
    .sum(axis=1)
    .replace(0, np.nan)
)
abs_ok["imp_ott"]  = abs_ok["beta_ott_shrunk"].abs()  / abs_sum
abs_ok["imp_app"]  = abs_ok["beta_app_shrunk"].abs()  / abs_sum
abs_ok["imp_arg"]  = abs_ok["beta_arg_shrunk"].abs()  / abs_sum
abs_ok["imp_putt"] = abs_ok["beta_putt_shrunk"].abs() / abs_sum

def _primary_skill(row):
    d = {
        "OTT": abs(row["beta_ott_shrunk"]),
        "APP": abs(row["beta_app_shrunk"]),
        "ARG": abs(row["beta_arg_shrunk"]),
        "PUTT": abs(row["beta_putt_shrunk"]),
    }
    return max(d, key=d.get)

abs_ok["primary_skill"] = abs_ok.apply(_primary_skill, axis=1)

abs_ok["predictability_pct"] = abs_ok["predictability"].rank(pct=True)
abs_ok["randomness_pct"]     = abs_ok["randomness"].rank(pct=True)
abs_ok["r2_pct"]             = abs_ok["r2"].rank(pct=True)

abs_ok.to_parquet(ABS_PARQUET, index=False)
abs_ok.to_csv(ABS_CSV, index=False)
print(f"\nOVERWROTE ABS:\n- {ABS_PARQUET}\n- {ABS_CSV}")
display(abs_ok.sort_values("predictability", ascending=False).head(20))

rel = abs_ok.copy()

rel_base = ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk","predictability","randomness","r2"]
for col in rel_base:
    m = rel[col].mean()
    s = rel[col].std(ddof=0)
    rel[f"{col}_z"] = (rel[col] - m) / s if s and s > 0 else np.nan
    rel[f"{col}_pct"] = rel[col].rank(pct=True)

rel.to_parquet(REL_PARQUET, index=False)
rel.to_csv(REL_CSV, index=False)
print(f"\nOVERWROTE REL:\n- {REL_PARQUET}\n- {REL_CSV}")
display(rel.sort_values("randomness", ascending=False).head(20))